# Stage 2 — CREsted enhancer code analysis FULL v2

This notebook is intentionally self-contained: config, helper functions, and execution are all included.

In [ ]:
from __future__ import annotations

import os
os.environ.setdefault("KERAS_BACKEND", "torch")

from pathlib import Path

import anndata as ad
import crested
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse

# =====================
# Config
# =====================
BASE = Path("/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt")
DATA = BASE / "data"
OUTDIR = BASE / "runs" / "out" / "stage2_enhancer_code_v2"
OUTDIR.mkdir(parents=True, exist_ok=True)

RUN_SPECIES = ["human"]  # first test human; then use ["human", "macaque"]
MODEL_LAYER = "model_prediction"
COMBINED_LAYER = "combined"
TOP_K = 2000              # if too slow/OOM, use 500 first
SPECIFICITY_METHOD = "gini"
CONTRIB_METHOD = "integrated_grad"  # strict tutorial fast option
RUN_CONTRIBUTION = True
RUN_TFMODISCO = True
REPORT = False            # set False if tomtom is unavailable, as tutorial recommends
MODISCO_WINDOW = 1000     # tutorial uses 1000; if seq_len=500 and this fails, change to 500
MAX_SEQLETS = 20000

DATASETS = {
    "human": {
        "adata": BASE / "runs" / "out" / "human_topics.h5ad",
        "model": BASE / "runs" / "out" / "deeptopic_human" / "final_model.keras",
        "genome_fasta": BASE.parent / "genomes" / "hg38" / "hg38.fa",
        "chrom_sizes": BASE.parent / "genomes" / "hg38" / "hg38.chrom.sizes",
        "annotation_candidates": [
            DATA / "3_topic_annotation_human_pycistopic.tsv",
            DATA / "topic_annotation_human_pycistopic.tsv",
            DATA / "human_topic_annotation_pycistopic.tsv",
        ],
        "qc_candidates": [
            DATA / "3_topic_qc_metrics_human.tsv",
            DATA / "topic_qc_metrics_human.tsv",
            DATA / "human_topic_qc_metrics.tsv",
        ],
    },
    "macaque": {
        "adata": BASE / "runs" / "out" / "macaque_topics.h5ad",
        "model": BASE / "runs" / "out" / "deeptopic_macaque" / "final_model.keras",
        "genome_fasta": BASE.parent / "yiquan" / "macaque" / "rheMac10.fa",
        "chrom_sizes": None,
        "annotation_candidates": [
            DATA / "3_topic_annotation_macaque_pycistopic.tsv",
            DATA / "topic_annotation_macaque_pycistopic.tsv",
            DATA / "macaque_topic_annotation_pycistopic.tsv",
        ],
        "qc_candidates": [
            DATA / "3_topic_qc_metrics_macaque.tsv",
            DATA / "topic_qc_metrics_macaque.tsv",
            DATA / "macaque_topic_qc_metrics.tsv",
        ],
    },
}

# =====================
# Helpers
# =====================
def first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None


def read_table_auto(path: Path | None):
    if path is None:
        return None
    sep = "\t" if str(path).endswith((".tsv", ".txt")) else ","
    return pd.read_csv(path, sep=sep)


def normalize_topic_name(x):
    s = str(x)
    if s.startswith("Topic"):
        return s
    try:
        return f"Topic{int(float(s))}"
    except Exception:
        return s


def load_table(candidates, label):
    path = first_existing(candidates)
    if path is None:
        print(f"[WARN] no {label} table found")
        return None
    print(f"[load {label}]", path)
    df = read_table_auto(path)
    if "topic" not in df.columns:
        first = df.columns[0]
        df = df.rename(columns={first: "topic"})
    df["topic"] = df["topic"].map(normalize_topic_name)
    return df


def attach_topic_annotation(adata, annot_df=None, qc_df=None):
    adata.obs["topic"] = [normalize_topic_name(x) for x in adata.obs_names]
    if annot_df is not None:
        add = annot_df.drop_duplicates("topic").set_index("topic")
        for col in add.columns:
            adata.obs[col] = adata.obs["topic"].map(add[col])
    if qc_df is not None:
        add = qc_df.drop_duplicates("topic").set_index("topic")
        for col in add.columns:
            new_col = col if col not in adata.obs.columns else f"{col}_qc"
            adata.obs[new_col] = adata.obs["topic"].map(add[col])
    print("[obs annotation columns]", list(adata.obs.columns))


def export_class_annotation(adata, out):
    adata.obs.to_csv(out, sep="\t")
    print("[save class annotation]", out)


def register_species_genome(cfg):
    fasta = cfg["genome_fasta"]
    chrom_sizes = cfg.get("chrom_sizes")
    if chrom_sizes is not None and Path(chrom_sizes).exists():
        genome = crested.Genome(str(fasta), str(chrom_sizes))
    else:
        genome = crested.Genome(str(fasta))
    crested.register_genome(genome)
    return genome


def add_predictions_to_layer(adata, model, layer_name):
    # Tutorial step: predictions = crested.tl.predict(adata, model); layer = predictions.T
    print("[predict] crested.tl.predict")
    pred = crested.tl.predict(adata, model=model)
    print("[predict shape]", pred.shape)
    adata.layers[layer_name] = pred.T
    print(f"[layer saved] adata.layers['{layer_name}'] shape =", adata.layers[layer_name].shape)


def make_combined_layer(adata, model_layer, combined_layer):
    # Tutorial: adata.layers['combined'] = (adata.X + adata.layers[model_layer]) / 2
    print("[combined layer]")
    x = adata.X.toarray() if sparse.issparse(adata.X) else np.asarray(adata.X)
    pred = adata.layers[model_layer]
    pred = pred.toarray() if sparse.issparse(pred) else np.asarray(pred)
    adata.layers[combined_layer] = (x + pred) / 2
    print(f"[layer saved] adata.layers['{combined_layer}'] shape =", adata.layers[combined_layer].shape)


def save_sort_filter_qc_plot(adata, out):
    try:
        crested.pl.qc.sort_and_filter_cutoff(
            adata,
            model_name=COMBINED_LAYER,
            cutoffs=[500, 1000, TOP_K],
            max_k=max(5000, TOP_K * 2),
        )
        plt.savefig(out, dpi=220, bbox_inches="tight")
        plt.close()
        print("[save fig]", out)
    except Exception as e:
        print("[WARN] sort/filter QC plot failed:", type(e).__name__, e)


def export_filtered_region_table(adata_filtered, out):
    df = adata_filtered.var.copy()
    df.index.name = "region"
    df.reset_index().to_csv(out, sep="\t", index=False)
    print("[save filtered regions]", out)


def run_one_species(tag, cfg):
    print(f"\n===== {tag} =====")
    species_out = OUTDIR / tag
    species_out.mkdir(parents=True, exist_ok=True)
    modisco_out = species_out / f"modisco_results_{tag}_top{TOP_K}"
    modisco_out.mkdir(parents=True, exist_ok=True)

    print("[load adata]", cfg["adata"])
    adata = ad.read_h5ad(cfg["adata"])
    print(adata)

    annot_df = load_table(cfg["annotation_candidates"], "annotation")
    qc_df = load_table(cfg["qc_candidates"], "QC")
    attach_topic_annotation(adata, annot_df, qc_df)
    export_class_annotation(adata, species_out / f"{tag}_class_annotation_used.tsv")

    print("[register genome]")
    register_species_genome(cfg)

    print("[load model]", cfg["model"])
    model = crested.utils.load_model(str(cfg["model"]))
    print(model)

    # Tutorial step 1: Store predictions for all regions.
    add_predictions_to_layer(adata, model, MODEL_LAYER)

    # Tutorial step 2: Calculate average of ground truth and predictions.
    make_combined_layer(adata, MODEL_LAYER, COMBINED_LAYER)

    # Tutorial QC plot for choosing top_k.
    save_sort_filter_qc_plot(adata, species_out / f"{tag}_sort_and_filter_cutoff_combined.png")

    # Tutorial step 3: Filter most informative regions per class.
    print(f"[filter] top_k={TOP_K}, method={SPECIFICITY_METHOD}, model_name={COMBINED_LAYER}")
    adata_filtered = crested.pp.sort_and_filter_regions_on_specificity(
        adata,
        model_name=COMBINED_LAYER,
        top_k=TOP_K,
        method=SPECIFICITY_METHOD,
        inplace=False,
    )
    print(adata_filtered)

    export_filtered_region_table(
        adata_filtered,
        species_out / f"{tag}_filtered_regions_top{TOP_K}_{SPECIFICITY_METHOD}.tsv",
    )
    filtered_h5ad = species_out / f"{tag}_adata_filtered_top{TOP_K}_{SPECIFICITY_METHOD}.h5ad"
    adata_filtered.write_h5ad(filtered_h5ad)
    print("[save]", filtered_h5ad)

    # Tutorial step 4: contribution scores for all classes.
    if RUN_CONTRIBUTION:
        print("[contribution_scores_specific] all classes, target_idx=None")
        crested.tl.contribution_scores_specific(
            input=adata_filtered,
            target_idx=None,
            model=model,
            output_dir=str(modisco_out),
            method=CONTRIB_METHOD,
        )

    # Tutorial step 5: TF-MoDISco-lite.
    if RUN_TFMODISCO:
        print("[motif db] crested.get_motif_db()")
        try:
            meme_db, motif_to_tf_file = crested.get_motif_db()
            print("[meme_db]", meme_db)
            print("[motif_to_tf_file]", motif_to_tf_file)
        except Exception as e:
            print("[WARN] crested.get_motif_db failed; running without meme_db")
            print(type(e).__name__, e)
            meme_db, motif_to_tf_file = None, None

        print("[tfmodisco]")
        kwargs = dict(
            window=MODISCO_WINDOW,
            output_dir=str(modisco_out),
            contrib_dir=str(modisco_out),
            report=REPORT,
            max_seqlets=MAX_SEQLETS,
        )
        if meme_db is not None:
            kwargs["meme_db"] = meme_db
        crested.tl.modisco.tfmodisco(**kwargs)

    return adata, adata_filtered


stage2_results = {}
for tag in RUN_SPECIES:
    stage2_results[tag] = run_one_species(tag, DATASETS[tag])
